# 09 · Diagnóstico — Hallazgos pendientes y Tooltips HTML en medidas

> **Tipo:** Solo lectura · **No ejecuta código** · **No modifica TMDL/PBIP** · Propósito: entender qué faltó, por qué pasó y cómo *podría* hacerse el tooltip HTML sin aplicarlo.

**Contexto:** Prueba técnica demanda UCSP (TUNI.pe 2020-2025). Capas `Bronce → Silver → Gold → Dashboard_parquet → PBIP`. Modelo estrella 5 dims + 2 hechos (450k mat / 166k ing). Este notebook resume los hallazgos detectados en la auditoría del 02-set y explica la viabilidad de tooltips HTML como *propuesta documental*.

| Estado | Significado |
|---|---|---|
| ✅ Correcto | Cumple requisito y está validado |
| ⚠️ Hallazgo | Funciona pero con deuda / artefacto |
| ❌ Pendiente | No cumple requisito o falta evidencia |

> **Regla de este notebook:** ninguna celda escribe en `data/` ni `dashboard/`. Solo markdown para leer y decidir si aplicar después.

---
## 1 · ETL — Lo que sí está sólido y lo que quedó como deuda

### ✅ Sólido (GATE 1 superado)
- **Contrato `data/schemas/*.json` como fuente de verdad** (`src/process_ingresantes.py:64`, `src/process_matriculados.py:90`): `columna_periodo`, `sep`, `encoding` y `clave_candidata` sin hardcodear. `validar_columnas()` falla si difiere.
- **Normalización:** `strip → UPPER → NFD → regex [\u0300-\u036f]` quita tildes en todas las `object/string`. En matriculados con Arrow `\p{Mn}` para no explotar 16GB (`process_matriculados.py:142`).
- **Streaming Polars <6GB:** matriculados procesado por archivo (`PERIODO+clave`, cada archivo = 1 semestre) + `sink_parquet` — evita OOM en `04_limpieza_matriculados.ipynb`.
- **0 nulos críticos, 0 huérfanos FK, 0 espacios** validados en `03_diagnostico_silver.ipynb` y `07_implementacion_gold.ipynb:496`.

### ⚠️ Hallazgo ETL-01 — 13,284 colisiones post-imputación (no eliminado)
- **Dónde:** `notebooks/04_limpieza_matriculados.ipynb:520` — K7 `PERIODO+INEI+SIU+GRUPO_1+GRUPO_3+LOCAL+GUID` = 0 en origen, **13,284 dup tras imputación** `CODIGO_GRUPO_1/3 → -1/SIN CLASIFICACION` (artefacto del mapa 520 programas).
- **Por qué pasó:** Ambición por cerrar Silver V2 rápido + mala gestión de tiempo: se reportó pero no se deduplicó de nuevo para no recalcular Gold.
- **Riesgo:** Bajo (0.07% de 17.3M), pero rompe unicidad K7 en `Gold/fact_matriculados.parquet`. Si el evaluador pide clave candidata estricta, es observación.
- **Cómo corregir (sin aplicar ahora):** `Gold/fact_matriculados.parquet` → `unique(subset=K7)` o re-imputar con clave K5 (sin grupos) como en `05_investigacion_clave`.

### ⚠️ Hallazgo ETL-02 — `Region_Sur` dual
- **Dónde:** `src/process_*.py:38` `REGION_SUR={6 deptos}` vs `dashboard/.../Dim_Zona.tmdl:24` que añade `MADRE DE DIOS` como Sur.
- **Por qué pasó:** Dos fuentes de verdad (código vs DAX). No se unificó catálogo.
- **Fix documental:** Unificar en `dim_ubicacion[Region_Sur]` y derivar `Dim_Zona` de ella, no de lista hardcodeada.

---
## 2 · Modelo Estrella Gold — Sólido pero desacoplado

### ✅ Sólido
- Star pura 5 dims (`dim_universidad 177`, `dim_programa ~500`, `dim_periodo 18`, `dim_ubicacion 98`, `dim_local 441`) + 2 hechos (`fact_ingresantes 166,666`, `fact_matriculados 450,448`) — `relationships.tmdl:1` 9 rel 1:N Single, `D007 CONFIRMED`.
- `DimPeriodo` conforme única `ANIO+SEMESTRE(NULL=ANUAL)` + `LABEL+TIPO` (6 anuales NULL +12 semestrales) respeta `P15/P16` no fabricar.
- `SK_* with_row_index(offset=1)` y `int64` TMDL ↔ `uint32` Parquet (`D006`).

### ❌ Hallazgo MOD-01 — Gold nunca llega al PBIP
- **Dónde:** `data/Gold/fact_matriculados.parquet` (17.3M granular GUID) vs `data/Dashboard_parquet/fact_matriculados_dashboard.parquet` (450k agregado `Conteo_Matriculados`). `dashboard/demanda_UCSP.SemanticModel/definition/model.tmdl:12` apunta a `Dashboard_parquet`.
- **Por qué pasó:** `F003` priorizó validar GATE 3 sobre agregado (más rápido) y dejó Gold como “propuesta conceptual” `notebooks/06`. Falta pipeline Gold→PBIP.
- **Riesgo:** Si el evaluador espera que Gold sea la fuente, ve duplicación y pregunta por qué hay 2 oros.
- **Fix:** Publicar Gold como fuente única o documentar en dossier que Dashboard_parquet es *agregado validado* y Gold es *granular para auditoría*.

### ⚠️ Hallazgo MOD-02 — D004/D005 PENDING pero implementados en visual
- `DECISION_REGISTER.md:D004` Norte/Centro pendiente, pero `Dim_Zona.tmdl:24` ya mapea Norte(8)/Centro(11)/Sur. `D005` Pregrado `CARRERA≡Pregrado` pendiente, pero dashboard muestra todos los `NIVEL_ACADEMICO` sin filtro.
- **Por qué:** Se implementó workaround UX para cumplir P1/P2 sin esperar negocio.
- **Fix:** Añadir filtro `NIVEL_ACADEMICO="CARRERA PROFESIONAL"` + disclaimer, y que `Dim_Zona` derive de `Region_Sur` validada.

---
## 3 · DAX — Donde está el artefacto que notaste (Matriculados vs Promedio)

### ✅ Sólido
- **Base:** `Total Ingresantes/Matriculados = SUM(Conteo_*)` `Medidas.tmdl:5` — `D002 técnico CONFIRMED` (0 nulos, buckets SEXO/RANGO disjuntos, 0 dup clave completa). No es personas únicas (D002 sem PENDING bien documentado).
- **UCSP fijo:** `CALCULATE([Total], CODIGO_INEI="260000062")` `Medidas.tmdl:20` con `SK 115` estable (D005).
- **Ranking:** `RANKX/TOPN(ALLSELECTED(SK_Programa), ..., Dense)` `Medidas.tmdl:38` respeta filtros `ANIO/Region_Sur/Gestión` (excluir blanks en visual).
- **YoY/Semestre:** aritmética `ANIO/SEMESTRE` por no tener tabla Date (`D006` SEMESTRE nullable).

### ⚠️ Hallazgo DAX-01 — Promedio Mercado con denominador de dimensión (el que viste)

| Año | Total mercado | Unis activas | Prom activo ★ | Prom dim (bug) | UCSP | UCSP vs activo | UCSP vs dim |
|---|---|---|---|---|---|---|---|
| 2020 | 2,410,123 | 169 | 14,261 | 13,617 | 15,980 | **+1,719 (1.12×)** | +2,363 |
| 2022 | 2,891,886 | 149 | 19,409 | 16,338 | 17,682 | **-1,727 (0.91×)** | +1,344 |
| **2024** | **3,101,317** | **141** | **21,995** | **17,522** | **18,664** | **-3,331 (0.85×)** | **+1,142** |
| 2025 | 3,294,737 | 140 | 23,534 | 18,614 | 18,978 | -4,556 (0.81×) | +364 |

- **Qué:** `Medidas.tmdl:118` `Total Universidades = DISTINCTCOUNT(dim_universidad[NOMBRE])` = **177 siempre**. El real es `DISTINCTCOUNT(fact[FK_Universidad])` = 169→140 con el tiempo (menos activas por año). En 2024 el bug baja el promedio **4,473** e invierte la lectura (UCSP parece siempre por encima).
- **Distribución nacional que lo explica:** media 99,780, mediana 18,012, UCSP 107,412 (rank 47/174) = 1.08× media pero 5.9× mediana; Top2 (César Vallejo 2.00M + UTP 1.94M =22% mercado) inflan media, 100 univs <20k la medianan. Con bug, UCSP siempre encima; corregido, cae por debajo nacional desde 2022 pero **sigue 1.33× por encima del Sur** (Sur 2024: 462k/33=14,022 vs UCSP 18,664, puesto 10/40 Sur).
- **Por qué pasó:** `ALLSELECTED(dim[NOMBRE])` se pensó que filtraba por hechos vía relación, pero `DISTINCTCOUNT(dim)` nunca filtra por existencia en fact. Junior + prisa, no se validó por año ni por Sur.
- **Cómo se corrige (solo documental, NO aplicado):**
  ```DAX
  Promedio Matrículas Mercado = AVERAGEX(ALLSELECTED('dim_universidad'[NOMBRE_ENTIDAD]), [Total Matrículas Mercado])
  -- AVERAGEX ignora blanks → promedia solo activas. Equivale a:
  -- DIVIDE([Total Mercado], CALCULATE(DISTINCTCOUNT(fact_matriculados_dashboard[FK_Universidad]), ALLSELECTED(dim[NOMBRE])))
  ```
  Ídem `Promedio Ingresantes Mercado` (138 activas, 0.90×). Se revertió a `DIVIDE(..., [Total Universidades])` para no romper PBIP (`git restore` del 02-set).
- **Qué llevar a la entrevista:** Muestra la tabla y di: “Con dim UCSP parecía por encima; corregido queda por debajo nacional pero lidera Sur. Lo dejo como variante `... (corregido - Activas)` para no romper el modelo productivo.”

### ⚠️ Hallazgo DAX-02 — Inconsistencia UCSP y YoY frágil
- `Total Ingresantes UCSP` `Medidas.tmdl:20` usa `[Total Ingresantes (Año)]` con `REMOVEFILTERS(SEMESTRE)` mientras `Total Matriculados UCSP` no — si el visual filtra por SEMESTRE, uno se corrige y otro no.
- `Variación % YoY` `Medidas.tmdl:26` y `Semestre Anterior` `Medidas.tmdl:32` usan `MAX(ANIO/SEMESTRE)`. Con multiselect (2023+2024) `MAX=2024` oculta 2023; en tabla por año cada fila usa el mismo MAX (requiere `SELECTEDVALUE`). Funciona si el slicer es single, falla con multi.
- **Fix documental:** Unificar patrón UCSP y usar `VAR añoActual = SELECTEDVALUE(dim_periodo[ANIO])` + `DATEADD` si se crea tabla Date.

---
## 4 · Visual / UX — Hallazgos menores

- **Filtros obligatorios operativos:** `Departamento` (25 depts, `dim_ubicacion`), `Region_Sur` (bool 23 prov) + `Dim_Zona` DAX y `Gestión` (PRIVADO 90/PUBLICO 87) en Single Direction sí afectan `ALLSELECTED` mercado — bien.
- **Top/Bottom 5:** `Medidas.tmdl:38` no filtra `BLANK/0` en DAX; depende de “excluir en visual”. Si el evaluador no lo hace, ve programas vacíos.
- **“Año completo = S1+S2” no implementado** (`M03/M12` CONDICIONADA por `D002` sem) — correcto no implementarlo, pero falta disclaimer en visual de matriculados semestral vs anual.
- **Tipos `uint32`/`float64` → `int64`** dejan 2 warnings `double` en `dim_programa` (`D006`), no bloqueante.

---
## 5 · Tooltips HTML en medidas — Cómo es posible *solo crearlos* sin aplicar (propuesta)

### 5.1 Qué ya existe en tu modelo (sin inventar)
- **Tabla `HTML1` `HTML1.tmdl:1`** con medidas que **retornan un string HTML** (`<div style='...'> ... " & FORMAT(...) & " ... </div>`). Ejemplos ya implementados y funcionando:
  - `HTML_KPI_Matriculas` `HTML1.tmdl:4` y `HTML_KPI_Ingresantes` `HTML1.tmdl:163` — tarjetas oscuras `171c23` con Total Mercado, UCSP y `Índice hipotético` (`DIVIDE(UCSP, Mercado)`).
  - `HTML_KPI_Cuota_Matriculas/Ingresantes` `HTML1.tmdl:322` — Market Share formateado `0.0%`.
  - `HTML_Ranking_Top3_UCSP` `HTML1.tmdl:506` — ranking Top3 + UCSP con `TOPN(3, ALLSELECTED(NOMBRE), [Total])` + `MAXX(EXCEPT(...))`.
  - `HTML_Tendencia_Matriculas/Ingresantes` `HTML1.tmdl:883` — gráfico SVG inline con `polyline` + `circle` para UCSP vs Promedio (`LineaUCSP`/`LineaMercado` con `YU/YM` escalados entre `Minimo/Rango` y `EscalaMin/Max`).
- **Visual que lo renderiza:** `htmlContent` custom visual `docs/htmlContent443BE3...pbiviz` (ya en `docs/`) — el PBIP lo usa para pintar ese HTML como tarjeta, no como visual nativo.

### 5.2 Cómo se crearía un tooltip HTML *solo como medida* (sin tocar páginas)

La idea no es tocar el tooltip nativo de Power BI (que es limitado), sino crear **medidas que devuelven HTML** y exponerlas donde el usuario hace hover. Hay 2 caminos, ambos *solo requieren crear medidas*, sin rehacer páginas:

#### Opción A — Medida HTML + visual HTML Content como tooltip (cero cambios de modelo)
1. **Crear medida que retorna HTML** (igual que `HTML1.tmdl:4`):
   ```DAX
   HTML_Tooltip_UCSP_vs_Promedio =
   VAR UCSP = [Total Matrículas UCSP]
   VAR Prom = [Promedio Matrículas Mercado]
   VAR Brecha = UCSP - Prom
   VAR Color = IF(Brecha>=0, "#16a34a", "#dc2626")
   RETURN
   "<div style='background:#0f172a;color:#fff;padding:8px 10px;border-radius:6px;font-family:Inter;font-size:10px;'>"
   & "<div style='color:#94a3b8;text-transform:uppercase;letter-spacing:0.06em;'>UCSP vs Promedio</div>"
   & "<div style='font-size:14px;font-weight:700;'>" & FORMAT(UCSP,"#,##0") & " vs " & FORMAT(Prom,"#,##0") & "</div>"
   & "<div style='color:" & Color & ";font-weight:700;'>" & FORMAT(Brecha,"+#,##0;-#,##0") & " (" & FORMAT(DIVIDE(Brecha,Prom),"+0.0%;-0.0%") & ")</div>"
   & "</div>"
   ```
2. **No se asigna a ninguna página:** la medida queda en la tabla `HTML1`, oculta, lista para ser arrastrada a un visual `HTML Content` configurado como **tooltip page** (Power BI: `Página → Permitir uso como información sobre herramientas`). El tooltip page contiene un único visual HTML Content cuyo campo es esa medida.
3. **Sin aplicar:** si no se crea la tooltip page ni se asigna a ningún visual, la medida existe pero no altera el reporte. Es 100% reversible (borrar medida).

#### Opción B — Medida HTML directa en tabla/matriz (aún más simple)
- La medida retorna HTML y se pone en una **columna de matriz** con el visual HTML Content en modo tabla. Al pasar el mouse, Power BI muestra el HTML renderizado como valor de celda — sin necesidad de tooltip page. También es solo crear medida.

### 5.3 Por qué *solo crear el documento* es la decisión correcta ahora
- `Medidas.tmdl` ya tiene 8 medidas HTML pesadas (`HTML_Tendencia_*` con 6 VAR por año + SVG). Añadir más sin validar rompe el PBIP (te pasó el 02-set al cambiar promedio). Crear las medidas en TMDL requiere `lineageTag` único y `PBIR` puede quedar corrupto si no se versiona.
- **Documentar el patrón** (qué medida, qué HTML, qué visual, cómo asignar como tooltip) te permite mostrar en entrevista que dominas la técnica (Deneb/HTML Content + `FORMAT` + `REMOVEFILTERS` + `ALLSELECTED`) sin arriesgar el PBIP que ya abre bien. En la sustentación abres `HTML1.tmdl:4` y enseñas el SVG trend como prueba.

### 5.4 Qué llevarías si decides aplicar después (checklist rápido, no ejecutar)
- [ ] Crear medida `HTML_Tooltip_*` en `HTML1` con `lineageTag` nuevo (no reutilizar `42012bfe`).
- [ ] Crear página `Tooltip_UCSP` (240×160), activar `Información sobre herramientas`, arrastrar visual HTML Content con esa medida.
- [ ] En visual principal (ej. barras por universidad), `Información sobre herramientas → Página = Tooltip_UCSP`.
- [ ] Probar con `ANIO=2024` y `Region_Sur` para ver UCSP -3,331 vs promedio (rojo) y Sur +4,? vs promedio (verde).

> **Conclusión:** Sí, es posible *solo crear* tooltips HTML como medidas sueltas — tu modelo ya lo hace. Este diagnóstico documenta el patrón y deja el *cómo* listo para cuando autorices el `git commit`, sin tocar nada hoy.

---
## 6 · Resumen — Qué mostrar en la entrevista con este diagnóstico

| Prioridad | Qué decir (30 seg cada uno) | Evidencia para abrir en sala |
|---|---|---|
| 1 | ETL y estrella sólidos, Gold desacoplado documentado | `03_diagnostico_silver`, `relationships.tmdl:1`, `GOLD/` vs `Dashboard_parquet` |
| 2 | **Promedio dim vs activo** — tabla 2020-2025, 2024 21,995 vs 17,522, UCSP 0.85× nacional pero 1.33× Sur | `Medidas.tmdl:118` + validación `polars` (no ejecutar, solo mostrar dossier) |
| 3 | **Tooltips HTML viables sin riesgo** — ya tienes 8 medidas `HTML1.tmdl:4` con SVG trend como prueba | Abrir `HTML1.tmdl:4` y `HTML_Tendencia_Matriculas:883` |
| 4 | Pregrado y Norte/Centro PENDING con fix documental | `DECISION_REGISTER.md:D004/D005` |

> **Actitud:** “Fui ambicioso, no medí tiempos, fallé en básicos (promedio, pregrado), los detecté con `polars` y traigo el fix sin romper el PBIP que ya abre. Eso es lo que aporto al equipo.”

---
*Notebook solo lectura — no modifica `data/` ni `dashboard/`.*
*Próximo paso si autorizas: crear `HTML_Tooltip_UCSP_vs_Promedio` en `HTML1` y página tooltip 240×160.*